# h3-motion-creator を Google Colab で動かす

[h3-motion-creator](https://github.com/KarakuriAgent/h3-motion-creator) 一式（ComfyUI と MiniMax H3・GEM-X・アプリ）を、Colab の GPU の実行環境の中に入れて動かします。
**上のセルから順に実行するだけ**です。手元に GPU が無くても使えます。

- GPU は RTX PRO 6000（G4）を想定しています。「ランタイム → ランタイムのタイプを変更」で GPU を選んでから始めてください
- 無料枠では、ノートブックの画面を通さず主に Web UI で操作することが Colab の規約で禁止されているので、コンピューティングユニットがある状態で使ってください。
- 実在の人物の動画は作らないでください。
- 作業データ（VRM・依頼・できた `.vrma`）は Google ドライブに残ります。ComfyUI・モデル（約 36GB）・GEM-X（重み約 6.4GB）はセッションごとに入れ直します
- 使うモデルとエージェントは、それぞれの配布元・提供元の決まりに従ってください

| セル | すること |
|---|---|
| 1〜3 | GPU を確かめる・ドライブをつなぐ・置き場を決める |
| 4-1〜4-7 | セットアップ（入っている物は飛ばすので、途中で止まったらそのセルからやり直せる） |
| 5〜6 | ComfyUI とアプリを起動し、アプリの画面を開く |
| 7 | エージェント（Claude Code・Codex CLI・Gemini CLI など）を Colab のターミナルに入れる |
| 8 | 後片付け |

In [ ]:
# @title 1. GPU を確かめる
# GPU の型番・メモリ・ドライバが出れば OK。出なければ「ランタイム → ランタイムのタイプを変更」で GPU を選ぶ
!nvidia-smi
!nvidia-smi --query-gpu=name,compute_cap,memory.total,driver_version --format=csv

In [ ]:
# @title 2. Google ドライブをつなぐ
# 作業データ（アプリの DATA_DIR）はドライブに置く。セッションが終わっても残り、次のセッションで続きができる
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title 3. 置き場を決める
# ComfyUI・GEM-X・アプリは /content の下（セッションごとに作り直す）。作業データはドライブの DRIVE_DIR の下の data/。
# TORCH_SOURCE: auto = Colab に入っている PyTorch が GPU に合えばそれを使い、合わなければ venv に cu128 / cu130 の PyTorch を入れる。
# Hugging Face のトークンが要るときは、左の鍵（シークレット）に HF_TOKEN を入れておく（無くても取れるモデルだけを使う）
import glob, json, os, pathlib, re, shutil, subprocess, sys, time, urllib.request

DRIVE_DIR = "/content/drive/MyDrive/h3-motion-creator"  # @param {type:"string"}
TORCH_SOURCE = "auto"  # @param ["auto", "colab", "cu128", "cu130"]

APP_DIR = "/content/h3-motion-creator"  # アプリ（このリポジトリの main）
COMFY_DIR = "/content/ComfyUI"          # ComfyUI（venv は ComfyUI/venv）
GEMX_DIR = "/content/GEM-X"             # GEM-X（venv は GEM-X/.venv）
BUILD_DIR = "/content/build"            # SageAttention のビルド
LOG_DIR = "/content/logs"               # ComfyUI・アプリ・ダウンロードのログ
ENV_FILE = "/content/h3-env.sh"         # ターミナル用の環境変数
DATA_DIR = f"{DRIVE_DIR}/data"          # アプリの作業データ（DATA_DIR）
CACHE_DIR = f"{DRIVE_DIR}/cache"        # ビルドした SageAttention の wheel（次のセッションはビルドを飛ばす）
REPO_URL = "https://github.com/KarakuriAgent/h3-motion-creator"
REPO_BRANCH = "main"
APP_PORT = 4170
COMFY_PORT = 8188
PY_SYS = sys.executable                 # Colab の Python（PyTorch が入っている）

if DRIVE_DIR.startswith("/content/drive/") and not os.path.ismount("/content/drive"):
    raise RuntimeError("ドライブがつながっていません。2. のセルを先に実行してください")
for d in (DATA_DIR, CACHE_DIR, LOG_DIR, BUILD_DIR):
    os.makedirs(d, exist_ok=True)

HOME = os.path.expanduser("~")
LOCAL_BIN = f"{HOME}/.local/bin"  # uv tool で入れる hf の置き場
if LOCAL_BIN not in os.environ["PATH"].split(":"):
    os.environ["PATH"] = f"{LOCAL_BIN}:{os.environ['PATH']}"
os.environ.update(APP_DIR=APP_DIR, COMFY_DIR=COMFY_DIR, GEMX_DIR=GEMX_DIR, BUILD_DIR=BUILD_DIR, LOG_DIR=LOG_DIR, DATA_DIR=DATA_DIR,
                  CACHE_DIR=CACHE_DIR, REPO_URL=REPO_URL, REPO_BRANCH=REPO_BRANCH, PY_SYS=PY_SYS, APP_URL=f"http://127.0.0.1:{APP_PORT}")
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN: シークレットから読んだ")
except Exception:
    print("HF_TOKEN: 無し（無くてよい）")

# ターミナル（7.）でも同じ置き場を使えるように
with open(ENV_FILE, "w") as f:
    f.write("# h3-motion-creator のノートブックが書いた\n")
    for k in ("APP_DIR", "COMFY_DIR", "GEMX_DIR", "DATA_DIR", "APP_URL"):
        f.write(f'export {k}="{os.environ[k]}"\n')
    f.write(f'export PATH="{LOCAL_BIN}:$PATH"\n')
bashrc = f"{HOME}/.bashrc"
src_line = f"[ -f {ENV_FILE} ] && . {ENV_FILE}"
if src_line not in (open(bashrc).read() if os.path.exists(bashrc) else ""):
    with open(bashrc, "a") as f:
        f.write(f"\n{src_line}\n")


def sh(script, env=None):
    """bash でスクリプトを動かし、出力を 1 行ずつ出す。失敗したら止める"""
    p = subprocess.Popen(["bash", "-c", "set -euo pipefail\n" + script], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, errors="replace", env={**os.environ, **(env or {})})
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"止まりました（終了コード {p.returncode}）。上の出力を見て直し、このセルをもう一度実行してください")


print(f"アプリ {APP_DIR}・ComfyUI {COMFY_DIR}・GEM-X {GEMX_DIR}・作業データ {DATA_DIR}")

In [ ]:
# @title 4-1. セットアップ: 道具（uv・git-lfs・ffmpeg・Node.js・hf）と PyTorch の決め方
# 入っている物は飛ばす。最後に、ComfyUI と GEM-X の venv で使う PyTorch と、SageAttention のビルドに使う CUDA を決めて出す。
# 4-4〜4-6 はここで作る関数を使うので、ランタイムを再起動したら 3. とこのセルを実行してから続ける
sh(r'''
command -v uv >/dev/null || "$PY_SYS" -m pip install -q uv
echo "uv: $(uv --version)"
need=()
command -v git-lfs >/dev/null || need+=(git-lfs)
{ command -v ffmpeg && command -v ffprobe; } >/dev/null || need+=(ffmpeg)
if [ ${#need[@]} -gt 0 ]; then
  echo "apt で入れる: ${need[*]}"
  apt-get -qq update
  DEBIAN_FRONTEND=noninteractive apt-get -qq install -y "${need[@]}" >/dev/null
fi
git lfs install --skip-repo >/dev/null
echo "git-lfs: $(git lfs version)"
echo "ffmpeg: $(ffmpeg -hide_banner -version | sed -n 1p)"
# Node.js 20 以上（古ければ 22 の LTS を /usr/local に入れる。PATH で Colab の物より先に来る）
node_major=$(node -p 'process.versions.node.split(".")[0]' 2>/dev/null || echo 0)
if [ "$node_major" -lt 20 ]; then
  base=https://nodejs.org/dist/latest-v22.x
  tarball=$(curl -fsSL "$base/SHASUMS256.txt" | grep -o -m1 'node-v[0-9.]*-linux-x64\.tar\.xz')
  echo "Node.js を入れる: $tarball"
  curl -fsSL "$base/$tarball" | tar -xJ -C /usr/local --strip-components=1 --no-same-owner --exclude='*.md' --exclude=LICENSE
  hash -r
  node_major=$(node -p 'process.versions.node.split(".")[0]')
  [ "$node_major" -ge 20 ] || { echo "Node.js 20 以上にできなかった（$(command -v node)）"; exit 1; }
fi
echo "node: $(node -v)（$(command -v node)）・npm: $(npm -v)"
# hf（Hugging Face の CLI。モデルを取る）
command -v hf >/dev/null || uv tool install -q 'huggingface_hub[hf_xet]'
echo "hf: $(command -v hf)"
''')


def run_out(cmd, cwd=None):
    """コマンドの出力（失敗なら空）"""
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, cwd=cwd)
    except OSError:
        return ""
    return r.stdout.strip() if r.returncode == 0 else ""


def ver(s):
    """'12.8' → (12, 8)。空なら (0, 0)"""
    return tuple(int(x) for x in re.findall(r"\d+", s or "")[:2]) or (0, 0)


TORCH_PROBE = "import json, torch; print(json.dumps({'version': torch.__version__, 'cuda': torch.version.cuda, 'archs': torch.cuda.get_arch_list(), 'file': torch.__file__}))"
PIN_PROBE = "import importlib.metadata as m\nfor n in ('torch', 'torchvision', 'torchaudio'):\n    try: print(f'{n}=={m.version(n)}')\n    except m.PackageNotFoundError: pass"


def torch_info(py):
    out = run_out([py, "-c", TORCH_PROBE])
    return json.loads(out.splitlines()[-1]) if out else None


def find_nvcc():
    """(nvcc の版, CUDA の根)。無ければ (None, None)"""
    cands = [f"{os.environ['CUDA_HOME']}/bin/nvcc"] if os.environ.get("CUDA_HOME") else []
    for n in cands + [shutil.which("nvcc") or "", "/usr/local/cuda/bin/nvcc"]:
        if n and os.path.isfile(n):
            m = re.search(r"release (\d+\.\d+)", run_out([n, "--version"]))
            if m:
                return m.group(1), os.path.dirname(os.path.dirname(os.path.realpath(n)))
    return None, None


def plan_torch():
    """どの PyTorch を使うか（colab / cu128 / cu130）と、SageAttention のビルドに要る CUDA を決める"""
    cap = run_out(["nvidia-smi", "-i", "0", "--query-gpu=compute_cap", "--format=csv,noheader"])
    if not cap:
        raise RuntimeError("GPU が見つかりません（1. のセル）")
    sm = "sm_" + cap.replace(".", "")
    need = (12, 8) if ver(cap) >= (10, 0) else (12, 0) if ver(cap) >= (9, 0) else (11, 8)  # この GPU に要る CUDA の版
    colab = torch_info(PY_SYS)
    nvcc, cuda_home = find_nvcc()
    colab_fits = bool(colab and colab["cuda"] and ver(colab["version"]) >= (2, 8) and ver(colab["cuda"]) >= need and sm in colab["archs"])
    src = TORCH_SOURCE
    if src == "auto":
        src = "colab" if colab_fits else "cu130" if ver(nvcc) >= (13, 0) else "cu128"
    torch_cuda = (colab or {}).get("cuda") if src == "colab" else {"cu128": "12.8", "cu130": "13.0"}[src]
    if not torch_cuda:
        raise RuntimeError("Colab の PyTorch に CUDA がありません。TORCH_SOURCE を cu128 か cu130 にして 3. からやり直す")
    # SageAttention のビルドには、PyTorch と大きい版が同じで、この GPU に足りる版の nvcc が要る。無ければ apt で入れる（4-5）
    toolkit = None if (nvcc and ver(nvcc)[0] == ver(torch_cuda)[0] and ver(nvcc) >= need) else torch_cuda
    return dict(gpu_cap=cap, sm=sm, colab_torch=colab and f"{colab['version']}（CUDA {colab['cuda']}）", colab_fits=colab_fits,
                nvcc=nvcc, cuda_home=cuda_home, source=src, torch_cuda=torch_cuda, cuda_toolkit=toolkit)


TORCH_PLAN = plan_torch()
print(json.dumps(TORCH_PLAN, ensure_ascii=False, indent=1))
print("PyTorch:", "Colab に入っている物をそのまま使う" if TORCH_PLAN["source"] == "colab" else f"venv に {TORCH_PLAN['source']} の物を入れる")


def ensure_venv(venv):
    """venv を作り、使う PyTorch をそろえて、(python, 版を固定する constraints のファイル) を返す"""
    py = f"{venv}/bin/python"
    if not os.path.exists(py):
        sh(f'uv venv --seed --system-site-packages --python "{PY_SYS}" "{venv}"')
    if TORCH_PLAN["source"] != "colab":
        t = torch_info(py)
        if not (t and t["file"].startswith(venv + "/") and t["cuda"] == TORCH_PLAN["torch_cuda"]):
            # uv は venv の外の物を見ないので、venv の中に torch 一式を入れる
            sh(f'uv pip install --python "{py}" torch torchvision torchaudio --index-url https://download.pytorch.org/whl/{TORCH_PLAN["source"]}')
    t = torch_info(py)
    if not t:
        raise RuntimeError(f"{py} で torch を読めません")
    pins = run_out([py, "-c", PIN_PROBE]).splitlines()
    constraints = f"{venv}/h3-constraints.txt"
    with open(constraints, "w") as f:
        f.write("\n".join(pins) + "\n")
    print(f"{py}: torch {t['version']}（CUDA {t['cuda']}・{os.path.dirname(t['file'])}）")
    return py, constraints


def ensure_cuda_toolkit():
    """SageAttention のビルドに使う CUDA（nvcc）を CUDA_HOME に。合う物が無ければ apt で入れる"""
    v = TORCH_PLAN["cuda_toolkit"]
    home = f"/usr/local/cuda-{v}" if v else TORCH_PLAN["cuda_home"]
    if v and not os.path.isfile(f"{home}/bin/nvcc"):
        sh(r'''
# CUDA_PKG（12-8 など）の nvcc と開発用のライブラリを apt で /usr/local/cuda-<版> に入れる（NVIDIA の apt の置き場が無ければ足す）
apt-get -qq update
if ! apt-cache show "cuda-nvcc-$CUDA_PKG" >/dev/null 2>&1; then
  . /etc/os-release
  curl -fsSLo /tmp/cuda-keyring.deb "https://developer.download.nvidia.com/compute/cuda/repos/ubuntu${VERSION_ID//./}/x86_64/cuda-keyring_1.1-1_all.deb"
  dpkg -i /tmp/cuda-keyring.deb >/dev/null
  apt-get -qq update
fi
echo "apt で入れる: cuda-nvcc-$CUDA_PKG cuda-libraries-dev-$CUDA_PKG"
DEBIAN_FRONTEND=noninteractive apt-get -qq install -y "cuda-nvcc-$CUDA_PKG" "cuda-libraries-dev-$CUDA_PKG" >/dev/null
''', env=dict(CUDA_PKG=v.replace(".", "-")))
    os.environ["CUDA_HOME"] = home
    if f"{home}/bin" not in os.environ["PATH"].split(":"):
        os.environ["PATH"] = f"{home}/bin:{os.environ['PATH']}"
    print("CUDA_HOME:", home, "・nvcc", find_nvcc()[0])

In [ ]:
# @title 4-2. セットアップ: ComfyUI の本体と、H3 のモデルのダウンロード（裏で）
# モデル（約 36GB。docs/setup.md §3 の表のとおり）は Hugging Face から並列に取る。裏で動くので、取っている間に次のセルへ進んでよい。
# 置き場に既にある物は飛ばす（途中で止まった物は、このセルをもう一度実行すると取り直す）
sh(r'''
if [ ! -d "$COMFY_DIR/.git" ]; then
  [ ! -e "$COMFY_DIR" ] || { echo "$COMFY_DIR が git の外のフォルダとしてある。消してからやり直す"; exit 1; }
  git clone --depth 1 https://github.com/Comfy-Org/ComfyUI "$COMFY_DIR"
fi
echo "ComfyUI: $(git -C "$COMFY_DIR" log -1 --format='%h %cd' --date=short)"
# MiniMax H3 のノードと BlockSparseAttention は本体に入っている（無ければ H3 に対応していない版）
for f in comfy_extras/nodes_minimax_h3.py comfy_extras/nodes_sparse_attention.py; do
  [ -f "$COMFY_DIR/$f" ] || { echo "ComfyUI に $f が無い"; exit 1; }
done
avail=$(df -BG --output=avail "$COMFY_DIR" | tail -n 1 | tr -dc '0-9')
echo "空き: ${avail}GB"
[ "$avail" -ge 60 ] || echo "注意: 空きが 60GB より少ない（H3 のモデル 36GB・GEM-X の重み 6.4GB・PyTorch など）"
''')
sh(r'''
cat > "$LOG_DIR/download_models.sh" <<'H3_EOF'
#!/usr/bin/env bash
# H3 のモデルを ComfyUI の models/ に並列で取る（ノートブックの 4-2 が書いて裏で動かす）。置き場にある物は飛ばす
set -uo pipefail
echo $$ > "$LOG_DIR/models.pid"
export HF_HUB_DISABLE_PROGRESS_BARS=1
M="$COMFY_DIR/models"
dl() {  # dl <リポジトリ> <リポジトリの中のパス> <models/ の下の置き場>
  local repo=$1 path=$2 sub=$3 name stage i
  name=$(basename "$path")
  stage="$M/.download/${repo//\//__}"
  if [ -s "$M/$sub/$name" ]; then echo "[あり] $sub/$name"; return 0; fi
  for i in 1 2 3; do
    if hf download "$repo" "$path" --local-dir "$stage" >/dev/null; then
      mkdir -p "$M/$sub" && mv -f "$stage/$path" "$M/$sub/$name" && echo "[取った] $sub/$name" && return 0
    fi
    echo "[やり直し $i] $repo $path"
    sleep 10
  done
  echo "[失敗] $repo $path"
  return 1
}
pids=()
dl Kijai/MiniMax-H3-experimental minimax_h3_fl2va_pruned_w4a8_mixed.safetensors diffusion_models & pids+=($!)
dl Kijai/MiniMax-H3-experimental loras/minimax_h3_ref_lora_rank_256_bf16.safetensors loras & pids+=($!)
dl lightx2v/Minimax-h3-Turbo minimax_h3_fl2v_turbo_4step_v1.3_768p_comfyui_bf16.safetensors loras & pids+=($!)
dl Kijai/MiniMax-H3-experimental minimax_h3_video_vae_int8_convrot.safetensors vae & pids+=($!)
dl Comfy-Org/MiniMax-H3 vae/minimax_h3_audio_vae_fp32.safetensors vae & pids+=($!)
dl Momoking/Qwen3-VL-32B-Heretic-MiniMax-H3-NVFP4 qwen3vl_32b_heretic_minimax_h3_nvfp4.safetensors text_encoders & pids+=($!)
rc=0
for p in "${pids[@]}"; do wait "$p" || rc=1; done
[ "$rc" -ne 0 ] || rm -rf "$M/.download"
echo "$rc" > "$LOG_DIR/models.done"
H3_EOF
pidf="$LOG_DIR/models.pid"
if [ -f "$pidf" ] && kill -0 "$(cat "$pidf")" 2>/dev/null && [ ! -f "$LOG_DIR/models.done" ]; then
  echo "ダウンロードはもう動いている（ログ: $LOG_DIR/models.log）"
  exit 0
fi
rm -f "$LOG_DIR/models.done" "$pidf"
setsid nohup bash "$LOG_DIR/download_models.sh" > "$LOG_DIR/models.log" 2>&1 < /dev/null &
echo "H3 のモデル（約 36GB）のダウンロードを裏で始めた（ログ: $LOG_DIR/models.log）。終わりは 4-7 で待つ"
''')

In [ ]:
# @title 4-3. セットアップ: アプリ（clone・npm ci・ヘッドレスのブラウザ・ビルド）
sh(r'''
if [ ! -d "$APP_DIR/.git" ]; then
  [ ! -e "$APP_DIR" ] || { echo "$APP_DIR が git の外のフォルダとしてある。消してからやり直す"; exit 1; }
  git clone --branch "$REPO_BRANCH" "$REPO_URL" "$APP_DIR"
fi
cd "$APP_DIR"
echo "アプリ: $(git log -1 --format='%h %cd' --date=short)"
[ -d node_modules ] || npm ci --no-audit --no-fund
# 三面図と確認用の動画を描くヘッドレスのブラウザ（入っていれば飛ばす）と、それが要る apt の物（セッションに 1 回）
npm run setup:browser
if [ ! -f "$LOG_DIR/.playwright-deps" ]; then
  npx playwright-core install-deps chromium
  touch "$LOG_DIR/.playwright-deps"
fi
if [ ! -f dist/server/server/index.js ] || [ ! -f dist/web/index.html ]; then npm run build; fi
echo "アプリの準備ができた: $APP_DIR"
''')

In [ ]:
# @title 4-4. セットアップ: ComfyUI の venv・PyTorch・カスタムノード
# venv は --system-site-packages で作り、Colab に入っている PyTorch（数 GB）を入れ直さずに使う（4-1 で合わないと決めたときだけ venv に入れる）。
# 足りない物は venv の中に入るので、Colab のノートブックの Python は変わらない。PyTorch は版を固定して、ほかの物の requirements で入れ替わらないようにする
if "TORCH_PLAN" not in globals():
    raise RuntimeError("先に 3. と 4-1 を実行してください（ランタイムを再起動したときも）")
COMFY_PY, COMFY_CONSTRAINTS = ensure_venv(f"{COMFY_DIR}/venv")
sh(r'''
cd "$COMFY_DIR"
pipi() { "$PY" -m pip install -q --progress-bar off -c "$CONSTRAINTS" "$@"; }
pipi -r requirements.txt
cd custom_nodes
clone() {
  [ -d "$2/.git" ] || git clone --depth 1 "$1" "$2"
  echo "$2: $(git -C "$2" log -1 --format='%h %cd' --date=short)"
  [ ! -f "$2/requirements.txt" ] || pipi -r "$2/requirements.txt"
}
clone https://github.com/kijai/ComfyUI-KJNodes ComfyUI-KJNodes                           # PathchSageAttentionKJ・MiniMaxH3MemoryEfficientSageAttentionPatch
clone https://github.com/xmarre/ComfyUI-Spectrum-MiniMax-H3 ComfyUI-Spectrum-MiniMax-H3   # SpectrumApplyMiniMaxH3
"$PY" -c "import torch; print('ComfyUI の torch:', torch.__version__, '・CUDA', torch.version.cuda, '・', torch.cuda.get_device_name(0))"
''', env=dict(PY=COMFY_PY, CONSTRAINTS=COMFY_CONSTRAINTS))

In [ ]:
# @title 4-5. セットアップ: SageAttention（ComfyUI の venv）
# ワークフローの MiniMaxH3MemoryEfficientSageAttentionPatch は SageAttention 2 の CUDA の関数を使う。PyPI の sageattention は 1 系（Triton だけ）で足りず、
# Linux 向けの 2 系の wheel は配られていないので、この GPU・PyTorch に合わせてソースからビルドする。
# できた wheel はドライブ（CACHE_DIR）に残し、次のセッションは PyTorch と GPU が同じならビルドを飛ばす
if "TORCH_PLAN" not in globals():
    raise RuntimeError("先に 3. と 4-1 を実行してください（ランタイムを再起動したときも）")
COMFY_PY, COMFY_CONSTRAINTS = f"{COMFY_DIR}/venv/bin/python", f"{COMFY_DIR}/venv/h3-constraints.txt"  # 4-4 で作った物
sage_probe = "import importlib.metadata as m, sageattention; print(m.version('sageattention'))"
have = run_out([COMFY_PY, "-c", sage_probe])
if ver(have) >= (2, 0):
    print(f"SageAttention {have} は入っている")
else:
    t = torch_info(COMFY_PY)
    cp = run_out([COMFY_PY, "-c", "import sys; print(f'cp{sys.version_info[0]}{sys.version_info[1]}')"])
    key = re.sub(r"[^A-Za-z0-9._-]", "_", f"torch{t['version']}-{TORCH_PLAN['sm']}-{cp}")
    wheel_dir = f"{CACHE_DIR}/wheels/sageattention/{key}"
    if not glob.glob(f"{wheel_dir}/sageattention-*.whl"):
        ensure_cuda_toolkit()
        sh(r'''
src="$BUILD_DIR/SageAttention"
[ -d "$src/.git" ] || git clone --depth 1 https://github.com/thu-ml/SageAttention "$src"
echo "SageAttention: $(git -C "$src" log -1 --format='%h %cd' --date=short)・CUDA_HOME=$CUDA_HOME・GPU $TORCH_CUDA_ARCH_LIST"
"$PY" -m pip install -q --progress-bar off -c "$CONSTRAINTS" ninja packaging wheel setuptools
jobs=$(nproc); [ "$jobs" -le 16 ] || jobs=16
export MAX_JOBS=$jobs EXT_PARALLEL=4
out="$BUILD_DIR/wheels"
rm -rf "$out"
echo "ビルドする（数分〜十数分）"
(cd "$src" && "$PY" -m pip wheel --no-build-isolation --no-deps -w "$out" .)
mkdir -p "$WHEEL_DIR"
cp "$out"/sageattention-*.whl "$WHEEL_DIR"/
echo "wheel をドライブに残した: $WHEEL_DIR"
''', env=dict(PY=COMFY_PY, CONSTRAINTS=COMFY_CONSTRAINTS, WHEEL_DIR=wheel_dir, TORCH_CUDA_ARCH_LIST=TORCH_PLAN["gpu_cap"]))
    wheel = sorted(glob.glob(f"{wheel_dir}/sageattention-*.whl"))[-1]
    sh(f'"{COMFY_PY}" -m pip install -q --no-deps --force-reinstall "{wheel}"')
    have = run_out([COMFY_PY, "-c", sage_probe])
    if not have:
        raise RuntimeError("import sageattention に失敗した")
    print(f"SageAttention {have} を入れた（{os.path.basename(wheel)}）")

In [ ]:
# @title 4-6. セットアップ: GEM-X（clone・venv・依存・重み 約 6.4GB）
# docs/setup.md §4 のとおり。soma-retargeter と detectron2 は要らないので入れない。venv は ComfyUI と同じ作り方（4-4）
if "TORCH_PLAN" not in globals():
    raise RuntimeError("先に 3. と 4-1 を実行してください（ランタイムを再起動したときも）")
sh(r'''
if [ ! -d "$GEMX_DIR/.git" ]; then
  [ ! -e "$GEMX_DIR" ] || { echo "$GEMX_DIR が git の外のフォルダとしてある。消してからやり直す"; exit 1; }
  git clone https://github.com/NVlabs/GEM-X "$GEMX_DIR"
fi
cd "$GEMX_DIR"
# 重み（下の Hugging Face の版）と合うコードの版に固定する
git checkout -q 32992550dba114c62243fb55e361311972dce8f9
# サブモジュールは要る 2 つだけ取る（soma-retargeter は要らない。その URL は SSH なので、鍵の無い所では取れない）。
# 中身が空のまま残った物も checkout し直し、SOMA の体の形（約 800MB）は git-lfs で取る
GIT_LFS_SKIP_SMUDGE=1 git -c 'url.https://github.com/.insteadOf=git@github.com:' submodule update --init --recursive third_party/sam-3d-body third_party/soma
git -C third_party/sam-3d-body checkout -q -f HEAD
git -C third_party/soma lfs pull
echo "GEM-X: $(git log -1 --format='%h %cd' --date=short)"
''')
GEMX_PY, GEMX_CONSTRAINTS = ensure_venv(f"{GEMX_DIR}/.venv")
sh(r'''
cd "$GEMX_DIR"
pipi() { "$PY" -m pip install -q --progress-bar off -c "$CONSTRAINTS" "$@"; }
pipi -e third_party/soma
pipi -e .
# scripts/install_env.sh の中身のうち要る物（detectron2 は推定に使わないので入れない。onnxruntime-gpu は人の検出に要る）
pipi cloudpickle fvcore iopath pycocotools braceexpand roma 'setuptools<75'
pipi onnxruntime-gpu
"$PY" -c "import onnxruntime as o; p = o.get_available_providers(); print('onnxruntime:', o.__version__, p); 'CUDAExecutionProvider' in p or print('注意: CUDAExecutionProvider が無い（CPU 版の onnxruntime とぶつかっていないか）')"
# SOMA の体（GEM-X は inputs/soma_assets から読む）と重み（Hugging Face の nvidia/GEM-X から約 6.4GB。ある物は飛ばす）
mkdir -p inputs
[ -e inputs/soma_assets ] || ln -s "$GEMX_DIR/third_party/soma/assets" inputs/soma_assets
# 重みは、このコードの版に合う Hugging Face の版に固定して取る（nvidia/GEM-X の main は新しい版の重みに入れ替わっている）
"$PY" -c "import functools, huggingface_hub as hh; hh.hf_hub_download = functools.partial(hh.hf_hub_download, revision='5b9a69d5ed4713ac4ba2a63b335e66db87279a3e'); from gem.utils import hf_utils as h; h.download_checkpoint(); h.download_vitpose_checkpoint(); h.download_sam3d_checkpoint(); h.download_mhr_model(); h.download_soma_data()"
for f in inputs/pretrained/gem_soma.ckpt inputs/checkpoints/vitpose/vitpose.pth inputs/checkpoints/sam-3d-body-dinov3/sam3d_body.ckpt inputs/mhr_data/mhr_model.pt inputs/soma_assets/SOMA_neutral.npz; do
  [ -e "$f" ] || { echo "無い: $f"; exit 1; }
done
"$PY" -c "import torch; print('GEM-X の torch:', torch.__version__, '・CUDA', torch.version.cuda, '・', torch.cuda.get_device_name(0))"
''', env=dict(PY=GEMX_PY, CONSTRAINTS=GEMX_CONSTRAINTS))

In [ ]:
# @title 4-7. セットアップ: H3 のモデルのダウンロードの終わりを待つ
done_file, pid_file = f"{LOG_DIR}/models.done", f"{LOG_DIR}/models.pid"


def alive(pid_path):
    try:
        os.kill(int(open(pid_path).read().strip()), 0)
        return True
    except (OSError, ValueError):
        return False


while not os.path.exists(done_file):
    if not alive(pid_file):
        time.sleep(3)  # 終わった直後かもしれない
        if not os.path.exists(done_file):
            raise RuntimeError(f"ダウンロードが止まっている（ログ: {LOG_DIR}/models.log）。4-2 をもう一度実行する")
        break
    got = sum(p.stat().st_size for p in pathlib.Path(f"{COMFY_DIR}/models").rglob("*") if p.is_file())
    print(f"{time.strftime('%H:%M:%S')}  {got / 1e9:.1f} / 約 36 GB", flush=True)
    time.sleep(30)
print(open(f"{LOG_DIR}/models.log").read())
if open(done_file).read().strip() != "0":
    raise RuntimeError("取れなかった物がある。4-2 をもう一度実行すると、取れなかった物だけ取り直す")
print("H3 のモデルがそろった")

In [ ]:
# @title 5. ComfyUI とアプリを起動する
# どちらも 127.0.0.1 だけで待ち受け、裏で動かす（ログは LOG_DIR）。もう動いていれば起動しない。
# 起動したら、アプリの設定に ComfyUI と GEM-X の場所を書き、環境チェックの結果を出す
def http(method, url, body=None, timeout=10):
    data = json.dumps(body).encode() if body is not None else None
    req = urllib.request.Request(url, data=data, method=method, headers={"Content-Type": "application/json"} if data else {})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read() or b"null")


def up(url):
    try:
        http("GET", url, timeout=5)
        return True
    except Exception:
        return False


def start(name, cmd, cwd, health, env=None, wait_s=900):
    if up(health):
        print(f"{name}: もう動いている")
        return
    log_path = f"{LOG_DIR}/{name}.log"
    log = open(log_path, "ab")
    p = subprocess.Popen(cmd, cwd=cwd, stdout=log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                         env={**os.environ, "PYTHONUNBUFFERED": "1", **(env or {})}, start_new_session=True)
    with open(f"{LOG_DIR}/{name}.pid", "w") as f:
        f.write(str(p.pid))
    t0 = time.time()
    while time.time() - t0 < wait_s:
        if p.poll() is not None:
            print(open(log_path, errors="replace").read()[-4000:])
            raise RuntimeError(f"{name} が止まった（ログ: {log_path}）")
        if up(health):
            print(f"{name}: 起動した（{time.time() - t0:.0f} 秒・ログ: {log_path}）")
            return
        time.sleep(3)
    raise TimeoutError(f"{name} が {wait_s} 秒で起動しない（ログ: {log_path}）")


COMFY_URL = f"http://127.0.0.1:{COMFY_PORT}"
APP_URL = f"http://127.0.0.1:{APP_PORT}"
start("comfyui", [f"{COMFY_DIR}/venv/bin/python", "main.py", "--listen", "127.0.0.1", "--port", str(COMFY_PORT)], COMFY_DIR, f"{COMFY_URL}/system_stats")
start("app", ["node", "dist/server/server/index.js"], APP_DIR, f"{APP_URL}/api/settings", env=dict(DATA_DIR=DATA_DIR, HOST="127.0.0.1", PORT=str(APP_PORT)), wait_s=120)

# 設定（DATA_DIR/settings.json）に場所を書く。python は空 = venv・.venv の中を自動で探す
http("PUT", f"{APP_URL}/api/settings", dict(comfyuiPath=COMFY_DIR, comfyuiUrl=COMFY_URL, gemxPath=GEMX_DIR, comfyuiPython="", gemxPython=""))
env = http("GET", f"{APP_URL}/api/env", timeout=600)
for i in env["items"]:
    print(f"{'OK  ' if i['state'] == 'ok' else '足りない' if i['state'] == 'missing' else i['state']}  {i['label']} — {i['detail']}")
    if i["state"] == "missing" and i.get("fix"):
        print(f"      直し方: {i['fix']}")
print("\n環境チェック:", "ready（すべて OK）" if env["ready"] else "足りない物がある（上の直し方を見る。ComfyUI のログ: " + f"{LOG_DIR}/comfyui.log）")

In [ ]:
# @title 6. アプリの画面を開く
# Colab のポートの転送（このノートブックを開いている Google アカウントだけが開ける URL）で、アプリの画面を新しいタブに開く。
# 画面は相対パスで API を呼ぶので、転送の URL のままで動く。リンクが開けなくなったら、このセルをもう一度実行する
from IPython.display import HTML, display
try:
    from google.colab.output import eval_js
    url = eval_js(f"google.colab.kernel.proxyPort({APP_PORT})")
    display(HTML(f'<p style="font-size:1.2em"><a href="{url}" target="_blank" rel="noopener">h3-motion-creator の画面を開く</a></p>'
                 f'<p>（{url}）</p>'))
except Exception as e:
    print("proxyPort で URL を取れなかった:", e)
    from google.colab import output
    output.serve_kernel_port_as_window(APP_PORT, anchor_text="h3-motion-creator の画面を開く")

In [ ]:
# @title 7. エージェントを入れる
# 選んだエージェントの CLI を入れる（npm i -g）。ログインと依頼は、下に出る案内のとおり Colab のターミナルで行う。
# 使うエージェントは、それぞれの提供元の規約に従ってください
AGENT = "Claude Code"  # @param ["Claude Code", "Codex CLI", "Gemini CLI", "入れない（自分で入れる）"]
from IPython.display import Markdown, display

AGENTS = {
    "Claude Code": dict(pkg="@anthropic-ai/claude-code", cmd="claude",
                        login="`claude` を起動し、ログインの仕方を選ぶ。出た URL をブラウザで開いてログインし、表示されたコードをターミナルに貼る"),
    "Codex CLI": dict(pkg="@openai/codex", cmd="codex",
                      login="`codex login --device-auth` を実行し、出た URL を開いてコードを入れてから `codex` を起動する"),
    "Gemini CLI": dict(pkg="@google/gemini-cli", cmd="gemini",
                       login="`NO_BROWSER=true gemini` で起動し、Google でのログインを選ぶ。出た URL を開いてログインし、表示されたコードを貼る"),
}
a = AGENTS.get(AGENT)
if a:
    sh(r'''
if command -v "$AGENT_CMD" >/dev/null; then
  echo "$AGENT_CMD は入っている: $(command -v "$AGENT_CMD")"
else
  npm i -g --no-audit --no-fund "$AGENT_PKG"
fi
"$AGENT_CMD" --version || true
''', env=dict(AGENT_PKG=a["pkg"], AGENT_CMD=a["cmd"]))
start_line = f"3. {a['login']}" if a else "3. 自分のエージェントの CLI を入れて（Node.js は 4-1 で入れた物が使える）、ログインして起動する"
display(Markdown(f"""### ターミナルでエージェントを動かす

1. 画面の左下の **ターミナル**（`>_`）を押して、Colab のターミナルを開く
2. `cd {APP_DIR}`
{start_line}
4. アプリの画面（6.）の「環境チェック／設定」が全部 OK なのを見て、「依頼」タブで作った**依頼文をコピーしてエージェントに貼る**
   （エージェントは同じ実行環境の中で動くので、依頼文の URL `http://127.0.0.1:{APP_PORT}` のままでアプリに届く）

ターミナルを開き直したら、`cd {APP_DIR}` からやり直す。各エージェントの規約に従って使ってください。
"""))

## 8. 後片付けと続き

- **作業データはドライブに残ります**（`DRIVE_DIR/data`: 読み込んだ VRM・依頼・元の動画・骨・`.vrma`・確認用の動画・設定）。次のセッションも 1. から順に実行すれば、同じ依頼の続きから使えます
- ComfyUI・H3 のモデル・GEM-X・エージェントのログインは、セッションを止めると消えます。次のセッションで入れ直します（SageAttention の wheel だけはドライブの `DRIVE_DIR/cache` に残るので、ビルドは飛ばせます）
- 手元に置きたい物（気に入った `.vrma`・比較の動画）は、アプリの確認画面からダウンロードするか、ドライブの `data/jobs/<依頼の id>/` から取る
- ComfyUI の出力（`/content/ComfyUI/output`）はドライブに残りません。要る物はセッションを止める前にダウンロードする
- 使い終わったら、下のセルで止めてから「ランタイム → ランタイムを接続解除して削除」で実行環境を消す（コンピューティングユニットの消費が止まる）

In [ ]:
# @title 8. 止める（ComfyUI とアプリを止め、ドライブへの書き込みを済ませる）
import signal
for name in ("app", "comfyui"):
    try:
        pid = int(open(f"{LOG_DIR}/{name}.pid").read().strip())
        os.killpg(pid, signal.SIGTERM)
        print(f"{name}: 止めた")
    except (OSError, ValueError):
        print(f"{name}: 動いていない")
from google.colab import drive
drive.flush_and_unmount()
print("ドライブへの書き込みを済ませて、つなぎを外した")